<a href="https://colab.research.google.com/github/zhuhroscar-tech/obs-stream-kit/blob/main/train_small_llm_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune a small LLM on my writing (Qwen2.5-0.5B) on Google Colab
Fine-tunes Qwen2.5-0.5B-Instruct on my own writing corpus, using full fine-tuning, partial layer freezing, or LoRA.

**Before you start:** Runtime → Change runtime type → T4 GPU (free tier is fine).

In [9]:
!pip install -q transformers datasets accelerate peft
!pip uninstall -y torchao

In [10]:
import torch, math
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
 DataCollatorForLanguageModeling, Trainer, TrainingArguments)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None - switch runtime to GPU!")

GPU: Tesla T4


## 1. Load data
My writing corpus from Google Drive, split into paragraphs (90% train / 10% test).

In [11]:
from google.colab import drive
drive.mount('/content/drive')   # click through the Google sign-in prompt

filename = "/content/drive/MyDrive/Oscar Zhu Info/my_writing_corpus_clean.txt"

import os
print(f"File size: {os.path.getsize(filename)/1024:.1f} KB")

raw = load_dataset("text", data_files=filename, split="train", sample_by="paragraph")
raw = raw.filter(lambda x: len(x["text"].strip()) > 0)
raw = raw.train_test_split(test_size=0.1, seed=42)
print(raw)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File size: 23.2 KB


Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/43 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 38
    })
    test: Dataset({
        features: ['text'],
        num_rows: 5
    })
})


## 2. Tokenize into chat-formatted examples
Each paragraph becomes a short chat (instruction + my writing as the reply), truncated to 512 tokens.

In [12]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
import random
random.seed(42)

SYSTEM_PROMPT = ("You are Oscar's writing assistant. Write in his voice: "
                 "concise, professional, finance-focused.")

USER_PROMPTS = [
    "Write a passage in my personal writing style.",
    "Write a paragraph the way I would.",
    "Draft this in my voice.",
    "Write something in my usual tone.",
]

def to_chat(batch):
    texts = []
    for t in batch["text"]:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": random.choice(USER_PROMPTS)},
            {"role": "assistant", "content": t.strip()},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return tokenizer(texts, truncation=True, max_length=512)

lm_data = raw.map(to_chat, batched=True, remove_columns=raw["train"].column_names)
print(lm_data)

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 38
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 5
    })
})


## 3. Load the pretrained model and choose a training method
Starts from pretrained weights. Set `METHOD` to "full", "freeze", or "lora".

In [13]:
METHOD = "lora"        # "full", "freeze", or "lora"
KEEP_LAYERS = 24       # layer config: 24 = all layers; fewer = smaller but weaker
UNFREEZE_LAST = 4      # for "freeze": how many final layers to train

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)

# Layer config
if KEEP_LAYERS < len(model.model.layers):
    model.model.layers = model.model.layers[:KEEP_LAYERS]
    model.config.num_hidden_layers = KEEP_LAYERS
    if getattr(model.config, "layer_types", None):
        model.config.layer_types = model.config.layer_types[:KEEP_LAYERS]

# Freeze layers
if METHOD == "freeze":
    for p in model.parameters():
        p.requires_grad = False
    for layer in model.model.layers[-UNFREEZE_LAST:]:
        for p in layer.parameters():
            p.requires_grad = True

# LoRA
elif METHOD == "lora":
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{METHOD}: training {trainable/1e6:.1f}M parameters")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

lora: training 8.8M parameters


## 4. Train

In [14]:
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)  # causal LM: labels = inputs shifted

args = TrainingArguments(
    output_dir="qwen-mystyle",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,   # saves GPU memory
    gradient_checkpointing_kwargs={"use_reentrant": False},   # needed for freeze/LoRA
    num_train_epochs=3,
    learning_rate={"full": 2e-5, "freeze": 5e-5, "lora": 2e-4}[METHOD],
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=1,
    report_to="none",
    eval_on_start=True,  # logs eval loss at step 0 = untouched base model
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_data["train"],
    eval_dataset=lm_data["test"],
    data_collator=collator,
)
def chat(user_prompt, system_prompt=SYSTEM_PROMPT, max_new_tokens=250, temperature=0.7):
    messages = []
    if system_prompt:                       # None → Qwen's default system prompt
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=temperature, top_p=0.9, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

p = "Write a passage in my personal writing style: why I want to work in finance."
before = chat(p)  # base model, before any training

trainer.train()

ev = trainer.evaluate()
print(f"Eval loss: {ev['eval_loss']:.3f}  |  Perplexity: {math.exp(ev['eval_loss']):.1f}")

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing_kwargs` is deprecated and will be removed in v6. Use `activation_checkpointing_kwargs` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Step,Training Loss,Validation Loss
0,No log,3.625916
15,3.369463,2.279866


Training Loss,Validation Loss,Step
3.369463,2.279866,15


Eval loss: 2.280  |  Perplexity: 9.8


## 5. Generate text

In [15]:
print(chat("Draft this in my voice: the opening of a cover letter for a summer analyst internship."))

print("BEFORE:\n", before, "\n\nAFTER:\n", chat(p))

Write to me about your academic experience and research interests.
BEFORE:
 In the grand tapestry of life, where dreams intertwine with reality, one thread seems to weave through the fabric of my existence - a dream that transcends mere ambition and embraces the pursuit of financial freedom. My passion for finance is not merely a desire; it's a beacon shining within me, guiding me towards the horizon of wealth and security.

The allure of finance lies in its ability to transform ordinary lives into extraordinary ones. It provides the tools to navigate the complexities of modern-day business, ensuring that every decision, no matter how small, contributes to the prosperity of our communities. This skillset empowers individuals like myself, who often find themselves at the mercy of economic cycles or underperforming their careers, to take control of their destiny by managing their finances wisely.

Moreover, as an AI language model, I have a unique perspective on finance, offering insight

## 6. Save (optionally to Google Drive so it survives the session)

In [16]:
final = model.merge_and_unload() if METHOD == "lora" else model
final.save_pretrained("qwen-mystyle-final")
tokenizer.save_pretrained("qwen-mystyle-final")

# Uncomment to copy to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r qwen-mystyle-final /content/drive/MyDrive/

# Reload later with:
# model = AutoModelForCausalLM.from_pretrained("qwen-mystyle-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen-mystyle-final/tokenizer_config.json',
 'qwen-mystyle-final/chat_template.jinja',
 'qwen-mystyle-final/tokenizer.json')